In [1]:
%config InlineBackend.figure_formats = ['svg']
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['FreeSans']

import pandas as pd, numpy as np, re, sys, torch, os
from types import SimpleNamespace
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed

# BCQ baseline
sys.path.insert(0, '../4_BCQ')
import model as _m_bcq
BCQ = _m_bcq.BCQ
from data import remap_rewards, EpisodicBuffer as EpisodicBufferBCQ

# BCQf factored
import importlib
sys.path.insert(0, '../4_BCQf')
import model as _m_bcqf
importlib.reload(_m_bcqf)
BCQf = _m_bcqf.BCQf
all_subactions_vec = _m_bcqf.all_subactions_vec
from data import EpisodicBuffer as EpisodicBufferBCQf

# Evaluation buffers
from evaluate import EpisodicBufferO, EpisodicBufferF, offline_evaluation_O, offline_evaluation_F

state_dim = 64
num_actions = 25
horizon = 20

In [2]:
# Load original (non-shifted) test data for both BCQ and BCQf
test_episodes_bcq = EpisodicBufferO(state_dim, num_actions, horizon)
test_episodes_bcq.load('../data/episodes+encoded_state+knn_pibs/test_data.pt')
test_episodes_bcq.reward = remap_rewards(
    test_episodes_bcq.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)

test_episodes_bcqf = EpisodicBufferF(state_dim, num_actions, horizon)
test_episodes_bcqf.load('../data/episodes+encoded_state+knn_pibs_factored/test_data.pt')
test_episodes_bcqf.reward = remap_rewards(
    test_episodes_bcqf.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)

print(f'BCQ  test: {len(test_episodes_bcq)} episodes')
print(f'BCQf test: {len(test_episodes_bcqf)} episodes')

Episodic Buffer loaded with 2892 episides.
Episodic Buffer loaded with 2892 episides.
BCQ  test: 2892 episodes
BCQf test: 2892 episodes


In [ ]:
# Load SHIFTED test data for both BCQ and BCQf
test_episodes_bcq_s = EpisodicBufferO(state_dim, num_actions, horizon)
test_episodes_bcq_s.load('../data/episodes+encoded_state+knn_pibs/shifted_test_data.pt')
test_episodes_bcq_s.reward = remap_rewards(
    test_episodes_bcq_s.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)

test_episodes_bcqf_s = EpisodicBufferF(state_dim, num_actions, horizon)
test_episodes_bcqf_s.load('../data/episodes+encoded_state+knn_pibs_factored/shifted_test_data.pt')
test_episodes_bcqf_s.reward = remap_rewards(
    test_episodes_bcqf_s.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)

print(f'BCQ  shifted test: {len(test_episodes_bcq_s)} episodes')
print(f'BCQf shifted test: {len(test_episodes_bcqf_s)} episodes')


In [3]:
def pareto2d(data):
    """Find Pareto frontier (maximizing both WIS and ESS)."""
    pts = np.array(data)
    mask = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        for j in range(len(pts)):
            if i != j and pts[j,0] >= pts[i,0] and pts[j,1] >= pts[i,1]:
                if pts[j,0] > pts[i,0] or pts[j,1] > pts[i,1]:
                    mask[i] = False
                    break
    return np.where(mask)[0]

def scan_versions(logdir, n_versions=40):
    """Scan all checkpoints from logdir, return DataFrame with val_wis, val_ess."""
    all_metrics = []
    for ver in range(n_versions):
        try:
            text = open(f'{logdir}/version_{ver}/hparams.yaml').read()
            thresh = float(re.search(r'threshold: ([0-9.]+)', text).group(1))
            seed = int(re.search(r'seed: ([0-9]+)', text).group(1))
            df = pd.read_csv(f'{logdir}/version_{ver}/metrics.csv')
            valid = df.dropna(subset=['val_wis', 'val_ess'])
            if len(valid) > 0:
                for _, row in valid.iterrows():
                    all_metrics.append({
                        'version': ver, 'threshold': thresh, 'seed': seed,
                        'val_wis': row['val_wis'], 'val_ess': row['val_ess'],
                        'iteration': row['iteration'],
                    })
        except Exception as e:
            pass  # skip missing versions
    return pd.DataFrame(all_metrics)

## 1. BCQ Baseline

In [4]:
logdir_bcq = '../4_BCQ/logs/mimic_dBCQ'
all_bcq = scan_versions(logdir_bcq)
print(f'BCQ: {len(all_bcq)} checkpoints from {all_bcq["version"].nunique()} versions')

# Pareto frontier
pts = np.column_stack([all_bcq['val_wis'].values, all_bcq['val_ess'].values])
pareto_idx = pareto2d(pts)
pareto_bcq = all_bcq.iloc[pareto_idx].copy().reset_index(drop=True)
print(f'Pareto frontier: {len(pareto_bcq)} non-dominated points')

# Tang's criterion: ESS >= 200, max WIS
ess_filtered = pareto_bcq[pareto_bcq['val_ess'] >= 200]
if len(ess_filtered) == 0:
    print('No models with val_ess >= 200, using all Pareto frontier')
    ess_filtered = pareto_bcq
best_bcq = ess_filtered.loc[ess_filtered['val_wis'].idxmax()]
print(f'Best BCQ: v{int(best_bcq["version"])}, tau={best_bcq["threshold"]}, seed={int(best_bcq["seed"])}')
print(f'  val_wis={best_bcq["val_wis"]:.2f}, val_ess={best_bcq["val_ess"]:.1f}, iter={int(best_bcq["iteration"])}')

BCQ: 3880 checkpoints from 40 versions
Pareto frontier: 24 non-dominated points
Best BCQ: v33, tau=0.75, seed=3
  val_wis=93.88, val_ess=208.3, iter=8475


In [5]:
# Evaluate all Pareto BCQ models on test set
bcq_results = []
for _, row in tqdm(pareto_bcq.iterrows(), total=len(pareto_bcq), desc='BCQ test'):
    ver = int(row['version'])
    best_iter = int(row['iteration'])
    step_files = sorted([int(f.stem.replace('step=', '')) for f in
        Path(logdir_bcq).glob(f'version_{ver}/step=*.ckpt')])
    if not step_files:
        continue
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f'{logdir_bcq}/version_{ver}/step={ckpt_step}.ckpt'
    
    model = BCQ.load_from_checkpoint(ckpt_path, map_location='cuda', weights_only=False)
    model.eval()
    test_loader = DataLoader(test_episodes_bcq, batch_size=len(test_episodes_bcq), shuffle=False)
    test_batch = next(iter(test_loader))
    test_batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in test_batch)
    w, e = model.offline_evaluation(test_batch, weighted=True, eps=0.01)
    bcq_results.append({
        'version': ver, 'threshold': row['threshold'], 'seed': row['seed'],
        'val_wis': row['val_wis'], 'val_ess': row['val_ess'],
        'best_iter': best_iter, 'ckpt_step': ckpt_step,
        'test_wis': w, 'test_ess': e,
    })

df_bcq = pd.DataFrame(bcq_results)
print(f'\nBCQ: {len(df_bcq)} models tested')
print(df_bcq.nlargest(10, 'test_wis')[['version','threshold','seed','best_iter','val_ess','test_wis','test_ess']].to_string(index=False))

BCQ test: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:45<00:00,  1.91s/it]


BCQ: 24 models tested
 version  threshold  seed  best_iter    val_ess  test_wis   test_ess
       9       0.01   4.0       5565 104.390709 99.999695  22.381416
       6       0.01   1.0       6920 128.577942 99.999416  12.285796
       9       0.01   4.0       2355  46.953667 99.992160   7.023497
       9       0.01   4.0       3410  98.674248 99.707562  50.264374
       2       0.00   2.0       2955  31.356632 98.207257 107.906029
       4       0.00   4.0       2255  38.891060 98.142703  92.850818
      18       0.10   3.0       1955 134.959412 97.894883 124.490386
      17       0.10   2.0       2755 143.783661 97.831918 127.628651
       4       0.00   4.0       2055  54.312630 97.719174  76.463735
      20       0.30   0.0        200 120.283607 97.478587 119.157307


In [ ]:
# SHIFTED — BCQ Pareto scan
logdir_bcq_s = '../4_BCQ/logs_shifted/mimic_dBCQ_shifted'
all_bcq_s = scan_versions(logdir_bcq_s)
print(f'BCQ shifted: {len(all_bcq_s)} checkpoints from {all_bcq_s["version"].nunique()} versions')

pts_s = np.column_stack([all_bcq_s['val_wis'].values, all_bcq_s['val_ess'].values])
pareto_idx_s = pareto2d(pts_s)
pareto_bcq_s = all_bcq_s.iloc[pareto_idx_s].copy().reset_index(drop=True)
print(f'Pareto frontier: {len(pareto_bcq_s)} non-dominated points')

ess_filtered_s = pareto_bcq_s[pareto_bcq_s['val_ess'] >= 200]
if len(ess_filtered_s) == 0:
    print('No models with val_ess >= 200, using all Pareto frontier')
    ess_filtered_s = pareto_bcq_s
best_bcq_s = ess_filtered_s.loc[ess_filtered_s['val_wis'].idxmax()]
print(f'Best BCQ shifted: v{int(best_bcq_s["version"])}, tau={best_bcq_s["threshold"]}, seed={int(best_bcq_s["seed"])}')
print(f'  val_wis={best_bcq_s["val_wis"]:.2f}, val_ess={best_bcq_s["val_ess"]:.1f}, iter={int(best_bcq_s["iteration"])}')


In [ ]:
# SHIFTED — Evaluate all Pareto BCQ models on test set
bcq_results_s = []
for _, row in tqdm(pareto_bcq_s.iterrows(), total=len(pareto_bcq_s), desc='BCQ shifted test'):
    ver = int(row['version'])
    best_iter = int(row['iteration'])
    step_files = sorted([int(f.stem.replace('step=', '')) for f in
        Path(logdir_bcq_s).glob(f'version_{ver}/step=*.ckpt')])
    if not step_files:
        continue
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f'{logdir_bcq_s}/version_{ver}/step={ckpt_step}.ckpt'
    
    model = BCQ.load_from_checkpoint(ckpt_path, map_location='cuda', weights_only=False)
    model.eval()
    test_loader = DataLoader(test_episodes_bcq_s, batch_size=len(test_episodes_bcq_s), shuffle=False)
    test_batch = next(iter(test_loader))
    test_batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in test_batch)
    w, e = model.offline_evaluation(test_batch, weighted=True, eps=0.01)
    bcq_results_s.append({
        'version': ver, 'threshold': row['threshold'], 'seed': row['seed'],
        'val_wis': row['val_wis'], 'val_ess': row['val_ess'],
        'best_iter': best_iter, 'ckpt_step': ckpt_step,
        'test_wis': w, 'test_ess': e,
    })

df_bcq_s = pd.DataFrame(bcq_results_s)
print(f'\nBCQ shifted: {len(df_bcq_s)} models tested')
print(df_bcq_s.nlargest(10, 'test_wis')[['version','threshold','seed','best_iter','val_ess','test_wis','test_ess']].to_string(index=False))


## 2. BCQf Factored

In [6]:
logdir_bcqf = '../4_BCQf/logs/mimic_dBCQf'
all_bcqf = scan_versions(logdir_bcqf)
print(f'BCQf: {len(all_bcqf)} checkpoints from {all_bcqf["version"].nunique()} versions')

# Pareto frontier
pts2 = np.column_stack([all_bcqf['val_wis'].values, all_bcqf['val_ess'].values])
pareto_idx2 = pareto2d(pts2)
pareto_bcqf = all_bcqf.iloc[pareto_idx2].copy().reset_index(drop=True)
print(f'Pareto frontier: {len(pareto_bcqf)} non-dominated points')

# Tang's criterion: ESS >= 200, max WIS
ess_filtered2 = pareto_bcqf[pareto_bcqf['val_ess'] >= 200]
if len(ess_filtered2) == 0:
    print('No models with val_ess >= 200, using all Pareto frontier')
    ess_filtered2 = pareto_bcqf
best_bcqf = ess_filtered2.loc[ess_filtered2['val_wis'].idxmax()]
print(f'Best BCQf: v{int(best_bcqf["version"])}, tau={best_bcqf["threshold"]}, seed={int(best_bcqf["seed"])}')
print(f'  val_wis={best_bcqf["val_wis"]:.2f}, val_ess={best_bcqf["val_ess"]:.1f}, iter={int(best_bcqf["iteration"])}')

BCQf: 3880 checkpoints from 40 versions
Pareto frontier: 21 non-dominated points
Best BCQf: v35, tau=0.9999, seed=0
  val_wis=94.71, val_ess=211.6, iter=7720


In [7]:
# Evaluate all Pareto BCQf models on test set
bcqf_results = []
for _, row in tqdm(pareto_bcqf.iterrows(), total=len(pareto_bcqf), desc='BCQf test'):
    ver = int(row['version'])
    best_iter = int(row['iteration'])
    step_files = sorted([int(f.stem.replace('step=', '')) for f in
        Path(logdir_bcqf).glob(f'version_{ver}/step=*.ckpt')])
    if not step_files:
        continue
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f'{logdir_bcqf}/version_{ver}/step={ckpt_step}.ckpt'
    
    model = BCQf.load_from_checkpoint(ckpt_path, map_location='cuda', weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec.to(model.device)
    test_loader = DataLoader(test_episodes_bcqf, batch_size=len(test_episodes_bcqf), shuffle=False)
    test_batch = next(iter(test_loader))
    test_batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in test_batch)
    w, e = model.offline_evaluation(test_batch, weighted=True, eps=0.01)
    bcqf_results.append({
        'version': ver, 'threshold': row['threshold'], 'seed': row['seed'],
        'val_wis': row['val_wis'], 'val_ess': row['val_ess'],
        'best_iter': best_iter, 'ckpt_step': ckpt_step,
        'test_wis': w, 'test_ess': e,
    })

df_bcqf = pd.DataFrame(bcqf_results)
print(f'\nBCQf: {len(df_bcqf)} models tested')
print(df_bcqf.nlargest(10, 'test_wis')[['version','threshold','seed','best_iter','val_ess','test_wis','test_ess']].to_string(index=False))

BCQf test: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:59<00:00,  2.84s/it]


BCQf: 21 models tested
 version  threshold  seed  best_iter    val_ess  test_wis   test_ess
       8       0.01   3.0       5965 146.243942 99.902118  13.619415
       1       0.00   1.0       2455  69.292519 99.247101  25.027046
       8       0.01   3.0       2355  37.937458 98.112187  72.075236
       4       0.00   4.0       2355  89.622787 97.924207 119.243548
       8       0.01   3.0       1755  42.941051 97.828405 113.221739
       3       0.00   3.0       1755  40.068081 97.789637 111.109643
      10       0.05   0.0       9775  97.981216 97.427818  94.459621
      10       0.05   0.0       7420 137.634811 97.345642 137.484011
      11       0.05   1.0       5565 128.809174 97.228631 130.792404
       2       0.00   2.0       5165 145.200394 97.062461 125.812425


In [ ]:
# SHIFTED — BCQf Pareto scan
logdir_bcqf_s = '../4_BCQf/logs_shifted/mimic_dBCQf_shifted'
all_bcqf_s = scan_versions(logdir_bcqf_s)
print(f'BCQf shifted: {len(all_bcqf_s)} checkpoints from {all_bcqf_s["version"].nunique()} versions')

pts2_s = np.column_stack([all_bcqf_s['val_wis'].values, all_bcqf_s['val_ess'].values])
pareto_idx2_s = pareto2d(pts2_s)
pareto_bcqf_s = all_bcqf_s.iloc[pareto_idx2_s].copy().reset_index(drop=True)
print(f'Pareto frontier: {len(pareto_bcqf_s)} non-dominated points')

ess_filtered2_s = pareto_bcqf_s[pareto_bcqf_s['val_ess'] >= 200]
if len(ess_filtered2_s) == 0:
    print('No models with val_ess >= 200, using all Pareto frontier')
    ess_filtered2_s = pareto_bcqf_s
best_bcqf_s = ess_filtered2_s.loc[ess_filtered2_s['val_wis'].idxmax()]
print(f'Best BCQf shifted: v{int(best_bcqf_s["version"])}, tau={best_bcqf_s["threshold"]}, seed={int(best_bcqf_s["seed"])}')
print(f'  val_wis={best_bcqf_s["val_wis"]:.2f}, val_ess={best_bcqf_s["val_ess"]:.1f}, iter={int(best_bcqf_s["iteration"])}')


In [ ]:
# SHIFTED — Evaluate all Pareto BCQf models on test set
bcqf_results_s = []
for _, row in tqdm(pareto_bcqf_s.iterrows(), total=len(pareto_bcqf_s), desc='BCQf shifted test'):
    ver = int(row['version'])
    best_iter = int(row['iteration'])
    step_files = sorted([int(f.stem.replace('step=', '')) for f in
        Path(logdir_bcqf_s).glob(f'version_{ver}/step=*.ckpt')])
    if not step_files:
        continue
    ckpt_step = min(step_files, key=lambda x: abs(x - best_iter))
    ckpt_path = f'{logdir_bcqf_s}/version_{ver}/step={ckpt_step}.ckpt'
    
    model = BCQf.load_from_checkpoint(ckpt_path, map_location='cuda', weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec.to(model.device)
    test_loader = DataLoader(test_episodes_bcqf_s, batch_size=len(test_episodes_bcqf_s), shuffle=False)
    test_batch = next(iter(test_loader))
    test_batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in test_batch)
    w, e = model.offline_evaluation(test_batch, weighted=True, eps=0.01)
    bcqf_results_s.append({
        'version': ver, 'threshold': row['threshold'], 'seed': row['seed'],
        'val_wis': row['val_wis'], 'val_ess': row['val_ess'],
        'best_iter': best_iter, 'ckpt_step': ckpt_step,
        'test_wis': w, 'test_ess': e,
    })

df_bcqf_s = pd.DataFrame(bcqf_results_s)
print(f'\nBCQf shifted: {len(df_bcqf_s)} models tested')
print(df_bcqf_s.nlargest(10, 'test_wis')[['version','threshold','seed','best_iter','val_ess','test_wis','test_ess']].to_string(index=False))


## 3. Bootstrap CI

In [11]:
# Tang's selection: val_ess >= 200, max val_wis
bcq_valid = pareto_bcq[pareto_bcq['val_ess'] >= 200]
if len(bcq_valid) == 0: bcq_valid = pareto_bcq
best_bcq_val = bcq_valid.loc[bcq_valid['val_wis'].idxmax()]

bcqf_valid = pareto_bcqf[pareto_bcqf['val_ess'] >= 200]
if len(bcqf_valid) == 0: bcqf_valid = pareto_bcqf
best_bcqf_val = bcqf_valid.loc[bcqf_valid['val_wis'].idxmax()]

best_bcq_row = df_bcq[(df_bcq['version']==int(best_bcq_val['version'])) & (abs(df_bcq['best_iter']-best_bcq_val['iteration'])<10)].iloc[0]
best_bcqf_row = df_bcqf[(df_bcqf['version']==int(best_bcqf_val['version'])) & (abs(df_bcqf['best_iter']-best_bcqf_val['iteration'])<10)].iloc[0]

print(f'Tang criterion: BCQ  v{int(best_bcq_val["version"])} tau={best_bcq_val["threshold"]} seed={int(best_bcq_val["seed"])} val_wis={best_bcq_val["val_wis"]:.2f} val_ess={best_bcq_val["val_ess"]:.1f}')
print(f'Tang criterion: BCQf v{int(best_bcqf_val["version"])} tau={best_bcqf_val["threshold"]} seed={int(best_bcqf_val["seed"])} val_wis={best_bcqf_val["val_wis"]:.2f} val_ess={best_bcqf_val["val_ess"]:.1f}')
# Tang paper exact: tau=0.5
bcq_tang = pareto_bcq[(pareto_bcq['threshold']==0.5) & (pareto_bcq['val_ess']>=200)]
if len(bcq_tang)==0: bcq_tang = pareto_bcq[pareto_bcq['threshold']==0.5]
best_bcq_tang = bcq_tang.loc[bcq_tang['val_wis'].idxmax()]

bcqf_tang = pareto_bcqf[(pareto_bcqf['threshold']==0.5) & (pareto_bcqf['val_ess']>=200)]
if len(bcqf_tang)==0: bcqf_tang = pareto_bcqf[pareto_bcqf['threshold']==0.5]
best_bcqf_tang = bcqf_tang.loc[bcqf_tang['val_wis'].idxmax()]

best_bcq_tang_row = df_bcq[(df_bcq['version']==int(best_bcq_tang['version'])) & (abs(df_bcq['best_iter']-best_bcq_tang['iteration'])<10)].iloc[0]
best_bcqf_tang_row = df_bcqf[(df_bcqf['version']==int(best_bcqf_tang['version'])) & (abs(df_bcqf['best_iter']-best_bcqf_tang['iteration'])<10)].iloc[0]

bcq_tang_ckpt = f'{logdir_bcq}/version_{int(best_bcq_tang_row["version"])}/step={int(best_bcq_tang_row["ckpt_step"])}.ckpt'
bcqf_tang_ckpt = f'{logdir_bcqf}/version_{int(best_bcqf_tang_row["version"])}/step={int(best_bcqf_tang_row["ckpt_step"])}.ckpt'

model_bcq_tang = BCQ.load_from_checkpoint(bcq_tang_ckpt, map_location='cuda', weights_only=False).eval()
model_bcqf_tang = BCQf.load_from_checkpoint(bcqf_tang_ckpt, map_location='cuda', weights_only=False).eval()
model_bcqf_tang.all_subactions_vec = all_subactions_vec.to(model_bcqf_tang.device)

def boot_bcq_tang(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcq_tang.offline_evaluation(batch, weighted=True, eps=0.01)

def boot_bcqf_tang(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcqf_tang.offline_evaluation(batch, weighted=True, eps=0.01)

print(f'Tang tau=0.5: BCQ  v{int(best_bcq_tang["version"])} seed={int(best_bcq_tang["seed"])} val_wis={best_bcq_tang["val_wis"]:.2f} val_ess={best_bcq_tang["val_ess"]:.1f}')
print(f'Tang tau=0.5: BCQf v{int(best_bcqf_tang["version"])} seed={int(best_bcqf_tang["seed"])} val_wis={best_bcqf_tang["val_wis"]:.2f} val_ess={best_bcqf_tang["val_ess"]:.1f}')

bcq_ckpt = f'{logdir_bcq}/version_{int(best_bcq_row["version"])}/step={int(best_bcq_row["ckpt_step"])}.ckpt'
bcqf_ckpt = f'{logdir_bcqf}/version_{int(best_bcqf_row["version"])}/step={int(best_bcqf_row["ckpt_step"])}.ckpt'

model_bcq = BCQ.load_from_checkpoint(bcq_ckpt, map_location='cuda', weights_only=False).eval()
model_bcqf = BCQf.load_from_checkpoint(bcqf_ckpt, map_location='cuda', weights_only=False).eval()
model_bcqf.all_subactions_vec = all_subactions_vec.to(model_bcqf.device)

# Bootstrap helpers
def boot_clinician(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    return buffer[idx][4].sum(axis=1).float().mean()

def boot_bcq(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcq.offline_evaluation(batch, weighted=True, eps=0.01)

def boot_bcqf(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcqf.offline_evaluation(batch, weighted=True, eps=0.01)

Tang criterion: BCQ  v33 tau=0.75 seed=3 val_wis=93.88 val_ess=208.3
Tang criterion: BCQf v35 tau=0.9999 seed=0 val_wis=94.71 val_ess=211.6
Tang tau=0.5: BCQ  v29 seed=4 val_wis=92.46 val_ess=226.3
Tang tau=0.5: BCQf v25 seed=0 val_wis=94.04 val_ess=229.4


In [ ]:
# SHIFTED — Tang's selection: val_ess >= 200, max val_wis
bcq_valid_s = pareto_bcq_s[pareto_bcq_s['val_ess'] >= 200]
if len(bcq_valid_s) == 0: bcq_valid_s = pareto_bcq_s
best_bcq_val_s = bcq_valid_s.loc[bcq_valid_s['val_wis'].idxmax()]

bcqf_valid_s = pareto_bcqf_s[pareto_bcqf_s['val_ess'] >= 200]
if len(bcqf_valid_s) == 0: bcqf_valid_s = pareto_bcqf_s
best_bcqf_val_s = bcqf_valid_s.loc[bcqf_valid_s['val_wis'].idxmax()]

best_bcq_row_s = df_bcq_s[(df_bcq_s['version']==int(best_bcq_val_s['version'])) & (abs(df_bcq_s['best_iter']-best_bcq_val_s['iteration'])<10)].iloc[0]
best_bcqf_row_s = df_bcqf_s[(df_bcqf_s['version']==int(best_bcqf_val_s['version'])) & (abs(df_bcqf_s['best_iter']-best_bcqf_val_s['iteration'])<10)].iloc[0]

print(f'SHIFTED Tang criterion: BCQ  v{int(best_bcq_val_s["version"])} tau={best_bcq_val_s["threshold"]} seed={int(best_bcq_val_s["seed"])} val_wis={best_bcq_val_s["val_wis"]:.2f} val_ess={best_bcq_val_s["val_ess"]:.1f}')
print(f'SHIFTED Tang criterion: BCQf v{int(best_bcqf_val_s["version"])} tau={best_bcqf_val_s["threshold"]} seed={int(best_bcqf_val_s["seed"])} val_wis={best_bcqf_val_s["val_wis"]:.2f} val_ess={best_bcqf_val_s["val_ess"]:.1f}')

# Tang tau=0.5 for shifted
bcq_tang_s = pareto_bcq_s[(pareto_bcq_s['threshold']==0.5) & (pareto_bcq_s['val_ess']>=200)]
if len(bcq_tang_s)==0: bcq_tang_s = pareto_bcq_s[pareto_bcq_s['threshold']==0.5]
best_bcq_tang_s = bcq_tang_s.loc[bcq_tang_s['val_wis'].idxmax()]

bcqf_tang_s = pareto_bcqf_s[(pareto_bcqf_s['threshold']==0.5) & (pareto_bcqf_s['val_ess']>=200)]
if len(bcqf_tang_s)==0: bcqf_tang_s = pareto_bcqf_s[pareto_bcqf_s['threshold']==0.5]
best_bcqf_tang_s = bcqf_tang_s.loc[bcqf_tang_s['val_wis'].idxmax()]

best_bcq_tang_row_s = df_bcq_s[(df_bcq_s['version']==int(best_bcq_tang_s['version'])) & (abs(df_bcq_s['best_iter']-best_bcq_tang_s['iteration'])<10)].iloc[0]
best_bcqf_tang_row_s = df_bcqf_s[(df_bcqf_s['version']==int(best_bcqf_tang_s['version'])) & (abs(df_bcqf_s['best_iter']-best_bcqf_tang_s['iteration'])<10)].iloc[0]

bcq_tang_ckpt_s = f'{logdir_bcq_s}/version_{int(best_bcq_tang_row_s["version"])}/step={int(best_bcq_tang_row_s["ckpt_step"])}.ckpt'
bcqf_tang_ckpt_s = f'{logdir_bcqf_s}/version_{int(best_bcqf_tang_row_s["version"])}/step={int(best_bcqf_tang_row_s["ckpt_step"])}.ckpt'

model_bcq_tang_s = BCQ.load_from_checkpoint(bcq_tang_ckpt_s, map_location='cuda', weights_only=False).eval()
model_bcqf_tang_s = BCQf.load_from_checkpoint(bcqf_tang_ckpt_s, map_location='cuda', weights_only=False).eval()
model_bcqf_tang_s.all_subactions_vec = all_subactions_vec.to(model_bcqf_tang_s.device)

print(f'SHIFTED Tang tau=0.5: BCQ  v{int(best_bcq_tang_s["version"])} seed={int(best_bcq_tang_s["seed"])} val_wis={best_bcq_tang_s["val_wis"]:.2f} val_ess={best_bcq_tang_s["val_ess"]:.1f}')
print(f'SHIFTED Tang tau=0.5: BCQf v{int(best_bcqf_tang_s["version"])} seed={int(best_bcqf_tang_s["seed"])} val_wis={best_bcqf_tang_s["val_wis"]:.2f} val_ess={best_bcqf_tang_s["val_ess"]:.1f}')

# Load shifted selected models
bcq_ckpt_s = f'{logdir_bcq_s}/version_{int(best_bcq_row_s["version"])}/step={int(best_bcq_row_s["ckpt_step"])}.ckpt'
bcqf_ckpt_s = f'{logdir_bcqf_s}/version_{int(best_bcqf_row_s["version"])}/step={int(best_bcqf_row_s["ckpt_step"])}.ckpt'

model_bcq_s = BCQ.load_from_checkpoint(bcq_ckpt_s, map_location='cuda', weights_only=False).eval()
model_bcqf_s = BCQf.load_from_checkpoint(bcqf_ckpt_s, map_location='cuda', weights_only=False).eval()
model_bcqf_s.all_subactions_vec = all_subactions_vec.to(model_bcqf_s.device)

# Bootstrap helpers for shifted
def boot_clinician_s(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    return buffer[idx][4].sum(axis=1).float().mean()

def boot_bcq_s(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcq_s.offline_evaluation(batch, weighted=True, eps=0.01)

def boot_bcqf_s(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcqf_s.offline_evaluation(batch, weighted=True, eps=0.01)

def boot_bcq_tang_s(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcq_tang_s.offline_evaluation(batch, weighted=True, eps=0.01)

def boot_bcqf_tang_s(i, buffer):
    n = len(buffer)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = buffer[idx]
    batch = tuple(t.cuda() if hasattr(t, 'cuda') else t for t in batch)
    return model_bcqf_tang_s.offline_evaluation(batch, weighted=True, eps=0.01)


In [12]:
print('Running 100 bootstraps...\n')

boot_clin = Parallel(n_jobs=1)(delayed(boot_clinician)(i, test_episodes_bcqf) for i in tqdm(range(100), desc='Clinician'))
boot_wis_bcq, boot_ess_bcq = zip(*Parallel(n_jobs=1)(delayed(boot_bcq)(i, test_episodes_bcq) for i in tqdm(range(100), desc='BCQ WIS')))
boot_wis_bcqf, boot_ess_bcqf = zip(*Parallel(n_jobs=1)(delayed(boot_bcqf)(i, test_episodes_bcqf) for i in tqdm(range(100), desc='BCQf WIS')))
boot_wis_bcq_tang, boot_ess_bcq_tang = zip(*Parallel(n_jobs=1)(delayed(boot_bcq_tang)(i, test_episodes_bcq) for i in tqdm(range(100), desc='BCQ tau=0.5')))
boot_wis_bcqf_tang, boot_ess_bcqf_tang = zip(*Parallel(n_jobs=1)(delayed(boot_bcqf_tang)(i, test_episodes_bcqf) for i in tqdm(range(100), desc='BCQf tau=0.5')))

Running 100 bootstraps...



BCQf tau=0.5: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:33<00:00,  2.73s/it]


In [ ]:
# SHIFTED — bootstraps
boot_clin_s = Parallel(n_jobs=1)(delayed(boot_clinician)(i, test_episodes_bcqf_s) for i in tqdm(range(100), desc='Clinician shifted'))
boot_wis_bcq_s, boot_ess_bcq_s = zip(*Parallel(n_jobs=1)(delayed(boot_bcq_s)(i, test_episodes_bcq_s) for i in tqdm(range(100), desc='BCQ shifted WIS')))
boot_wis_bcqf_s, boot_ess_bcqf_s = zip(*Parallel(n_jobs=1)(delayed(boot_bcqf_s)(i, test_episodes_bcqf_s) for i in tqdm(range(100), desc='BCQf shifted WIS')))
boot_wis_bcq_tang_s, boot_ess_bcq_tang_s = zip(*Parallel(n_jobs=1)(delayed(boot_bcq_tang_s)(i, test_episodes_bcq_s) for i in tqdm(range(100), desc='BCQ shifted tau=0.5')))
boot_wis_bcqf_tang_s, boot_ess_bcqf_tang_s = zip(*Parallel(n_jobs=1)(delayed(boot_bcqf_tang_s)(i, test_episodes_bcqf_s) for i in tqdm(range(100), desc='BCQf shifted tau=0.5')))


## 4. Results

In [13]:
clin_wis = test_episodes_bcqf.reward.sum(axis=1).mean().item()
clin_ess = len(test_episodes_bcqf)

print(f'{"="*60}')
print(f'Clinician    WIS: {clin_wis:.2f} +/- {np.std(boot_clin):.2f}  ESS: {clin_ess:.0f}')
print(f'Baseline BCQ WIS: {best_bcq_row["test_wis"]:.2f} +/- {np.std(boot_wis_bcq):.2f}  ESS: {best_bcq_row["test_ess"]:.1f} +/- {np.std(boot_ess_bcq):.1f}')
print(f'Factored BCQ WIS: {best_bcqf_row["test_wis"]:.2f} +/- {np.std(boot_wis_bcqf):.2f}  ESS: {best_bcqf_row["test_ess"]:.1f} +/- {np.std(boot_ess_bcqf):.1f}')
print(f'{"="*60}')
print()
print('Selection criterion (Tang): val_ess >= 200, max val_wis')
print(f'  BCQ:  v{int(best_bcq_val["version"])} tau={best_bcq_val["threshold"]} seed={int(best_bcq_val["seed"])}')
print(f'  BCQf: v{int(best_bcqf_val["version"])} tau={best_bcqf_val["threshold"]} seed={int(best_bcqf_val["seed"])}')
print(f'Tang tau=0.5 BCQ : WIS: {best_bcq_tang_row["test_wis"]:.2f} +/- {np.std(boot_wis_bcq_tang):.2f}  ESS: {best_bcq_tang_row["test_ess"]:.1f} +/- {np.std(boot_ess_bcq_tang):.1f}')
print(f'Tang tau=0.5 BCQf: WIS: {best_bcqf_tang_row["test_wis"]:.2f} +/- {np.std(boot_wis_bcqf_tang):.2f}  ESS: {best_bcqf_tang_row["test_ess"]:.1f} +/- {np.std(boot_ess_bcqf_tang):.1f}')
print()
print(f'Tang (2022) paper:')
print(f'  Clinician:    WIS=90.29 +/- 0.51  ESS=2894')
print(f'  Baseline BCQ: WIS=90.44 +/- 2.44  ESS=178.32 +/- 11.42')
print(f'  Factored BCQ: WIS=91.62 +/- 2.12  ESS=178.32 +/- 11.96')

Clinician    WIS: 90.25 +/- 0.57  ESS: 2892
Baseline BCQ WIS: 95.61 +/- 1.26  ESS: 212.9 +/- 11.5
Factored BCQ WIS: 94.79 +/- 1.42  ESS: 224.7 +/- 12.7

Selection criterion (Tang): val_ess >= 200, max val_wis
  BCQ:  v33 tau=0.75 seed=3
  BCQf: v35 tau=0.9999 seed=0
Tang tau=0.5 BCQ : WIS: 94.72 +/- 1.51  ESS: 200.0 +/- 11.8
Tang tau=0.5 BCQf: WIS: 93.68 +/- 1.42  ESS: 231.2 +/- 12.3

Tang (2022) paper:
  Clinician:    WIS=90.29 +/- 0.51  ESS=2894
  Baseline BCQ: WIS=90.44 +/- 2.44  ESS=178.32 +/- 11.42
  Factored BCQ: WIS=91.62 +/- 2.12  ESS=178.32 +/- 11.96


In [ ]:
# SHIFTED results
clin_wis_s = test_episodes_bcqf_s.reward.sum(axis=1).mean().item()
clin_ess_s = len(test_episodes_bcqf_s)

print(f'\n{"=== SHIFTED RESULTS ===".center(60)}')
print(f'{"="*60}')
print(f'Clinician    WIS: {clin_wis_s:.2f} +/- {np.std(boot_clin_s):.2f}  ESS: {clin_ess_s:.0f}')
print(f'Baseline BCQ WIS: {best_bcq_row_s["test_wis"]:.2f} +/- {np.std(boot_wis_bcq_s):.2f}  ESS: {best_bcq_row_s["test_ess"]:.1f} +/- {np.std(boot_ess_bcq_s):.1f}')
print(f'Factored BCQ WIS: {best_bcqf_row_s["test_wis"]:.2f} +/- {np.std(boot_wis_bcqf_s):.2f}  ESS: {best_bcqf_row_s["test_ess"]:.1f} +/- {np.std(boot_ess_bcqf_s):.1f}')
print(f'{"="*60}')
print()
print('Selection criterion (Tang): val_ess >= 200, max val_wis')
print(f'  BCQ:  v{int(best_bcq_val_s["version"])} tau={best_bcq_val_s["threshold"]} seed={int(best_bcq_val_s["seed"])}')
print(f'  BCQf: v{int(best_bcqf_val_s["version"])} tau={best_bcqf_val_s["threshold"]} seed={int(best_bcqf_val_s["seed"])}')
print(f'Tang tau=0.5 BCQ : WIS: {best_bcq_tang_row_s["test_wis"]:.2f} +/- {np.std(boot_wis_bcq_tang_s):.2f}  ESS: {best_bcq_tang_row_s["test_ess"]:.1f} +/- {np.std(boot_ess_bcq_tang_s):.1f}')
print(f'Tang tau=0.5 BCQf: WIS: {best_bcqf_tang_row_s["test_wis"]:.2f} +/- {np.std(boot_wis_bcqf_tang_s):.2f}  ESS: {best_bcqf_tang_row_s["test_ess"]:.1f} +/- {np.std(boot_ess_bcqf_tang_s):.1f}')
